# 📊 Stock Price 21-Day EMA Analysis
## Data Analysis with Python Assignment

**Objective:** Calculate the 21-day Exponential Moving Average (EMA) for 5 tech stocks and identify which has the highest momentum.

**Dataset:** `stock_prices.csv` containing ~126 trading days for AAPL, MSFT, GOOGL, AMZN, META

## 📁 Step 1: Upload Your Data File

Upload `stock_prices.csv` using the file upload button in Colab

In [ ]:
# Uncomment and run this cell to upload files in Google Colab
from google.colab import files
uploaded = files.upload()
# After upload, the file will be in the current directory

## 📦 Step 2: Import Required Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("✓ Libraries imported successfully!")

## 📖 Step 3: Load and Explore the Data

In [ ]:
# Load the CSV file
df = pd.read_csv("stock_prices.csv")

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 10 rows:")
print(df.head(10))
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Unique tickers: {df['Ticker'].unique()}")
print(f"Trading days per ticker: {df.groupby('Ticker').size().iloc[0]}")

## 🧮 Step 4: Calculate 21-Day EMA

**EMA Formula:**
```
multiplier (k) = 2 / (span + 1) = 2 / 22 ≈ 0.0909
EMA[0] = Close[0]
EMA[i] = Close[i] × k + EMA[i-1] × (1 - k)
```

**Important:** Use `adjust=False` in Pandas `ewm()` to get the standard recursive EMA formula.

In [ ]:
# Calculate 21-day EMA for each ticker
df["EMA_21"] = df.groupby("Ticker")["Close_Price"].transform(
    lambda x: x.ewm(span=21, adjust=False).mean()
)

print("\n✓ EMA calculation complete!")
print(f"\nSample EMA values:")
print(df[["Date", "Ticker", "Close_Price", "EMA_21"]].head(25))

## 🏆 Step 5: Find the Highest EMA on the Last Trading Day

In [ ]:
# Get the last date
last_date = df["Date"].max()
print(f"Last trading date: {last_date}")

# Filter for last day only
last_day = df[df["Date"] == last_date].copy()
last_day = last_day.sort_values("EMA_21", ascending=False)

print(f"\n21-day EMA values on {last_date}:")
print(last_day[["Ticker", "Close_Price", "EMA_21"]].to_string(index=False))

# Find the winner
winner_ticker = last_day.iloc[0]["Ticker"]
winner_ema = last_day.iloc[0]["EMA_21"]

print("\n" + "=" * 70)
print("🏆 FINAL ANSWER 🏆")
print("=" * 70)
print(f"\nHighest 21-day EMA on {last_date}:")
print(f"Ticker: {winner_ticker}")
print(f"EMA Value: {winner_ema:.2f}")
print(f"\n📋 Submission format: {winner_ema:.2f}, {winner_ticker}")
print("=" * 70)

## 📊 Step 6: Visualizations

### 6.1 Individual Stock Charts with EMA

In [ ]:
# Plot all stocks with their EMA
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Stock Prices and 21-Day EMA', fontsize=16, fontweight='bold')

tickers = df['Ticker'].unique()
for idx, ticker in enumerate(tickers):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    ticker_data = df[df['Ticker'] == ticker].copy()
    ticker_data = ticker_data.sort_values('Date')
    
    ax.plot(ticker_data.index, ticker_data['Close_Price'], 
            label='Close Price', alpha=0.7, linewidth=1.5)
    ax.plot(ticker_data.index, ticker_data['EMA_21'], 
            label='21-day EMA', linewidth=2, color='red')
    
    ax.set_title(f'{ticker}', fontweight='bold')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Remove extra subplot
axes[2, 1].remove()

plt.tight_layout()
plt.show()

### 6.2 EMA Comparison Bar Chart

In [ ]:
# Bar chart of final EMA values
plt.figure(figsize=(10, 6))
colors = ['gold' if t == winner_ticker else 'steelblue' for t in last_day['Ticker']]
bars = plt.bar(last_day['Ticker'], last_day['EMA_21'], color=colors, alpha=0.8, edgecolor='black')

# Add value labels on bars
for bar, value in zip(bars, last_day['EMA_21']):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'${value:.2f}',
             ha='center', va='bottom', fontweight='bold')

plt.title(f'21-Day EMA Comparison - {last_date}', fontsize=14, fontweight='bold')
plt.xlabel('Ticker', fontsize=12, fontweight='bold')
plt.ylabel('EMA Value ($)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 6.3 EMA Trends Over Time

In [ ]:
# EMA trend over time for all stocks
plt.figure(figsize=(12, 6))
for ticker in tickers:
    ticker_data = df[df['Ticker'] == ticker].copy()
    ticker_data = ticker_data.sort_values('Date')
    
    linestyle = '--' if ticker == winner_ticker else '-'
    linewidth = 3 if ticker == winner_ticker else 1.5
    
    plt.plot(ticker_data.index, ticker_data['EMA_21'], 
             label=ticker, linewidth=linewidth, linestyle=linestyle)

plt.title('21-Day EMA Trends - All Stocks', fontsize=14, fontweight='bold')
plt.xlabel('Trading Day', fontsize=12)
plt.ylabel('EMA Value ($)', fontsize=12)
plt.legend(title='Ticker', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ✅ Summary

**What is EMA?**
- Exponential Moving Average gives more weight to recent prices
- More responsive to price changes than Simple Moving Average (SMA)
- Used by traders to identify momentum and trend direction

**Key Findings:**
- META has the highest 21-day EMA on the last trading day
- This indicates META has the strongest recent price momentum
- The EMA smooths out short-term fluctuations while highlighting trends